# Store chunk embeddings in ChromaDB

Store production BGE-M3 450-token chunk embeddings in ChromaDB while preserving parent article metadata.

In [ ]:
import json
from pathlib import Path

import chromadb
import numpy as np

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
EMBEDDING_DIR = DATA_DIR / 'processed' / 'embeddings'
CHROMA_DIR = DATA_DIR / 'processed' / 'chroma_db'
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
CHROMA = chromadb.PersistentClient(path=str(CHROMA_DIR))
BATCH_SIZE = 256

DATASETS = {
    'recursive-450-tokens': ('chunks_bge_m3_recursive_450_tokens', 'legal_chunks_bge_m3_recursive_450_tokens'),
}

def load_dataset(prefix, level):
    vector_path = EMBEDDING_DIR / f'{prefix}.npy'
    metadata_path = EMBEDDING_DIR / f'{prefix}_metadata.jsonl'
    manifest_path = EMBEDDING_DIR / f'{prefix}_manifest.json'
    missing = [str(path) for path in (vector_path, metadata_path, manifest_path) if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing embedding output: ' + ', '.join(missing))
    vectors = np.load(vector_path).astype('float32')
    records = [json.loads(line) for line in metadata_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected_count = manifest.get('article_count', manifest.get('chunk_count'))
    if vectors.ndim != 2 or len(records) != vectors.shape[0] or expected_count != vectors.shape[0]:
        raise ValueError(f'{level} vectors, metadata, and manifest counts do not match')
    if manifest.get('embedding_dimension') != vectors.shape[1]:
        raise ValueError(f'{level} manifest dimension does not match vectors')
    return vectors, records, manifest

def scalar_metadata(record, model_name):
    result = {'embedding_model': model_name}
    for key, value in record.items():
        if key in {'text', 'search_text'} or value is None:
            continue
        result[key] = value if isinstance(value, (str, int, float, bool)) else json.dumps(value, ensure_ascii=False)
    return result

def store_dataset(model_key):
    prefix, collection_name = DATASETS[model_key]
    vectors, records, manifest = load_dataset(prefix, 'chunk')
    collection = CHROMA.get_or_create_collection(name=collection_name, metadata={'model': manifest['model'], 'hnsw:space': 'cosine'})
    for start in range(0, len(records), BATCH_SIZE):
        batch = records[start:start + BATCH_SIZE]
        ids = [str(record['chunk_id']) for record in batch]
        collection.upsert(ids=ids, embeddings=vectors[start:start + BATCH_SIZE].tolist(), documents=[str(record.get('text', '')) for record in batch], metadatas=[scalar_metadata(record, manifest['model']) for record in batch])
    return {'level': 'chunk', 'model': model_key, 'collection': collection_name, 'stored': collection.count(), 'dimension': int(vectors.shape[1])}

print('ChromaDB:', CHROMA_DIR)
print('Datasets:', DATASETS)

In [ ]:
results = []
for model_key, (prefix, _) in DATASETS.items():
    if not all((EMBEDDING_DIR / f'{prefix}{suffix}').exists() for suffix in ('.npy', '_metadata.jsonl', '_manifest.json')):
        print(f'Skipping {model_key}: embedding files are not available yet')
        continue
    results.append(store_dataset(model_key))

print(json.dumps(results, ensure_ascii=False, indent=2))